In [ ]:
from KirKode.Batch_version.scaled_MSE.B1_batch_scaling import (
    NeuralNetworkModel,
    Losses_MSE,
    Losses_MSE_Batched,
    RiemannianMetric,
    RiemannianMetricBatched,
    Brownian_sampler,
    Brownian_sampler_batched,
    Gaussian_sampler,
)
from C_NN import Net_2_Layer, Train_Net
import numpy as np
import torch
import matplotlib.pyplot as plt

In [ ]:
def compute_scaled_hessian(loss_fn, theta, lam):
    """Hessian of (lam * loss_fn.Loss) at theta. Returns (dim, dim)."""
    theta_d = theta.detach().clone().requires_grad_(True)
    scaled_loss = lambda th: lam * loss_fn.Loss(th)
    H = torch.autograd.functional.hessian(scaled_loss, theta_d)
    return H.detach()


def psd_inverse(H, alpha=1e-2):
    """Stable inverse of a (near-)PSD matrix via Cholesky, with eigen-clip fallback."""
    dim = H.shape[0]
    H_sym = 0.5 * (H + H.T)
    I = torch.eye(dim, dtype=H.dtype, device=H.device)
    try:
        L = torch.linalg.cholesky(H_sym + alpha * I)
        Sigma = torch.cholesky_inverse(L)
    except RuntimeError:
        evals, evecs = torch.linalg.eigh(H_sym)
        evals_clipped = torch.clamp(evals, min=alpha)
        Sigma = (evecs * (1.0 / evals_clipped)) @ evecs.T
    return 0.5 * (Sigma + Sigma.T)


def laplace_init_diag(sampler, Sigma):
    """Set sampler.rho so softplus(rho)^2 = diag(Sigma)."""
    assert sampler.covariance_type == "diag"
    sd = torch.sqrt(torch.clamp(torch.diag(Sigma), min=1e-30))
    rho_new = torch.log(torch.expm1(sd.clamp(min=1e-6)))
    with torch.no_grad():
        sampler.rho.copy_(rho_new.to(sampler.rho.dtype))


def laplace_init_full(sampler, Sigma):
    """Set sampler.rho_diag, sampler.A_off so A^T A = Sigma, A upper-tri."""
    assert sampler.covariance_type == "full"
    L = torch.linalg.cholesky(Sigma)          # lower-tri
    A = L.T.contiguous()                       # upper-tri, positive diag
    diag_A = torch.diag(A)
    rho_diag_new = torch.log(torch.expm1(diag_A.clamp(min=1e-6)))
    A_off_new = A - torch.diag(diag_A)
    with torch.no_grad():
        sampler.rho_diag.copy_(rho_diag_new.to(sampler.rho_diag.dtype))
        sampler.A_off.copy_(A_off_new.to(sampler.A_off.dtype))


def make_laplace_sampler(loss_fn, theta, lam, dim, n_samples,
                         covariance_type="diag", alpha=1e-2):
    """Returns a Gaussian_sampler with mu=theta, Sigma=(lam*H_MSE + alpha I)^{-1}."""
    H = compute_scaled_hessian(loss_fn, theta, lam=lam)
    Sigma = psd_inverse(H, alpha=alpha)

    sampler = Gaussian_sampler(
        dim=dim, n_samples=n_samples,
        init_mu=theta.detach().clone(),
        init_sigma=0.1,                  
        covariance_type=covariance_type,
        dtype=torch.float64,
    )
    if covariance_type == "diag":
        laplace_init_diag(sampler, Sigma)
    else:
        laplace_init_full(sampler, Sigma)
    return sampler, Sigma

# Batched

In [ ]:

def compute_scaled_hessian_batched(loss_b, theta, lam):
   
    theta_d = theta.detach().clone().requires_grad_(True)
    S = loss_b.S
    model = loss_b.model
 
    H_sum = None
    for idx in loss_b.batch_idx:
        x_b = loss_b.x[idx]
        y_b = loss_b.y_data[idx]
 
        def batch_loss(th):
            y_hat = model.forward_theta(th, x_b)
            return torch.mean((y_b - y_hat) ** 2)
 
        H_b = torch.autograd.functional.hessian(batch_loss, theta_d)
        H_sum = H_b if H_sum is None else H_sum + H_b
 
    H_avg = H_sum / float(S)
    return (lam * H_avg).detach()
 
 
def make_laplace_sampler_batched(loss_b, theta, lam, dim, n_samples,
                                 covariance_type="diag", alpha=1e-2):
  
    H = compute_scaled_hessian_batched(loss_b, theta, lam=lam)
    Sigma = psd_inverse(H, alpha=alpha)
 
    sampler = Gaussian_sampler(
        dim=dim, n_samples=n_samples,
        init_mu=theta.detach().clone(),
        init_sigma=0.1,                   
        covariance_type=covariance_type,
        dtype=torch.float64,
    )
    if covariance_type == "diag":
        laplace_init_diag(sampler, Sigma)
    else:
        laplace_init_full(sampler, Sigma)
    return sampler, Sigma
 